# পাঠ ১৩ - কগনী নলেজ গ্রাফের সাথে এজেন্ট মেমরি


## সেটআপ

এই নোটবুকটি দেখাবে কিভাবে [**Cognee**](https://www.cognee.ai/) জ্ঞান গ্রাফ এবং **Microsoft Agent Framework** (MAF) ব্যবহার করে স্থায়ী মেমরিসহ একটি বুদ্ধিমান **কোডিং অ্যাসিস্ট্যান্ট** তৈরি করা যায়।

Cognee অবিন্যাসিত টেক্সটকে একটি কাঠামোবদ্ধ, অনুসন্ধানযোগ্য জ্ঞান গ্রাফে রূপান্তরিত করে যা ভেক্টর এম্বেডিং দ্বারা সমর্থিত — আপনার এজেন্টকে সমৃদ্ধ, সম্পর্ক-সচেতন দীর্ঘমেয়াদি মেমরি প্রদান করে।

### আপনি যা শিখবেন
1. **জ্ঞান গ্রাফ তৈরি করুন**: ডেভেলপার প্রোফাইল এবং সেরা অনুশীলনগুলি কাঠামোবদ্ধ, অনুসন্ধানযোগ্য জ্ঞানে রুপান্তর করুন।
2. **Cognee কে MAF এর সাথে সংযুক্ত করুন**: MAF এজেন্টকে Cognee এর জ্ঞান গ্রাফ অনুসন্ধান করার জন্য `@tool` ফাংশনগুলি ব্যবহার করুন।
3. **সেশন-সচেতন আলোচনা**: একই সেশনে একাধিক প্রশ্নের মধ্যে প্রেক্ষাপট বজায় রাখুন।
4. **দীর্ঘমেয়াদি মেমরি**: সেশন পেরিয়ে গুরুত্বপূর্ণ জ্ঞান সংরক্ষণ করুন এবং নতুন আলোচনায় এটি পুনরুদ্ধার করুন।

### প্রয়োজনীয়তা
- পাইথন ৩.৯+
- লোকালি Redis চলমান (`docker run -d -p 6379:6379 redis`) সেশন ম্যানেজমেন্টের জন্য
- একটি LLM API কী (যেমন OpenAI) — `.env` ফাইলে `LLM_API_KEY` সেট করুন
- `.env` ফাইলে `CACHING=true` (Cognee সেশনের জন্য প্রয়োজন)
- একটি Azure AI Foundry প্রকল্প যেখানে একটি চ্যাট মডেল ডিপ্লয় করা আছে
- `.env` ফাইলে `AZURE_AI_PROJECT_ENDPOINT` এবং `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI এ অনুমোদিত (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## এজেন্ট মেমোরির ধরন

এই নোটবুকটি মূল লেসন ১৩ নোটবুকের একই তিন ধরনের মেমোরি অন্বেষণ করে, তবে দীর্ঘমেয়াদী মেমোরি ব্যাকএন্ড হিসাবে Cognee ব্যবহার করে:

| মেমোরি টাইপ | পদ্ধতি | জীবনকাল |
|---|---|---|
| **ওয়ার্কিং** | `agent.create_session()` (MAF) | একক কথোপকথন |
| **স্বল্পমেয়াদী** | Cognee সেশন ক্যাশ (Redis) | একক সেশন |
| **দীর্ঘমেয়াদী** | Cognee জ্ঞান গ্রাফ + ভেক্টর | স্থায়ী |

### Cognee এর মেমোরি আর্কিটেকচার
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## কগনি স্টোরেজ প্রস্তুত করুন


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Part 1 — জ্ঞানের ভাণ্ডার তৈরি করা

আমরা আমাদের কোডিং সহকারীর জন্য বিস্তৃত জ্ঞানের ভাণ্ডার তৈরির জন্য তিন ধরনের ডেটা গ্রহণ করি:

1. **ডেভেলপার প্রোফাইল** — ব্যক্তিগত দক্ষতা এবং প্রযুক্তিগত পটভূমি
2. **পাইথন সেরা অনুশীলনসমূহ** — ব্যবহারিক নির্দেশিকা সহ পাইথনের জেন
3. **ঐতিহাসিক কথোপকথন** — ডেভেলপার এবং AI সহকারীদের মধ্যে অতীত প্রশ্নোত্তর সেশনসমূহ


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## জ্ঞান গ্রাফ কল্পনা করুন

কগনি একটি ইন্টারেক্টিভ HTML ভিজ্যুয়ালাইজেশন প্রদর্শন করতে পারে যা এটি নিষ্কাশিত অংকের এবং সম্পর্কের।


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memify দিয়ে স্মৃতি সমৃদ্ধ করুন

`memify()` জ্ঞান গ্রাফ বিশ্লেষণ করে এবং বুদ্ধিমান নিয়ম তৈরি করে — ধারণাগুলির মধ্যে নিদর্শন, সেরা অনুশীলন এবং সম্পর্ক চিন্হিত করে।


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## অংশ ২ — Cognee টুলস সহ MAF এজেন্ট

এখন আমরা একটি MAF এজেন্ট তৈরি করবো যা `@tool` ফাংশনের মাধ্যমে Cognee এর জ্ঞানের গ্রাফে প্রশ্ন করতে পারে। এটি এজেন্টকে সেশনের মাধ্যমে কথোপকথনের প্রসঙ্গ বজায় রেখে গ্রাফ-সচেতন অর্থবোধক অনুসন্ধানের সম্পূর্ণ শক্তি কাজে লাগানোর সুযোগ দেয়।


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## সেশন সহ ওয়ার্কার মেমোরি

`AgentSession` (যা `agent.create_session()` মাধ্যমে তৈরি হয়) একটি সেশনের মধ্যে ওয়ার্কার মেমোরি প্রদান করে। এজেন্ট পূর্ববর্তী বার্তাগুলিকে উল্লেখ করতে পারে একই সাথে Cognee এর দীর্ঘমেয়াদী জ্ঞান গ্রাফকেও প্রশ্ন করতে পারে।


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## নতুন সেশন — দীর্ঘমেয়াদি স্মৃতি টিকে থাকে

একটি নতুন সেশন শুরু করলে ওয়ার্কিং মেমরি পরিষ্কার হয়ে যায়, কিন্তু Cognee জ্ঞান গ্রাফ এখনও উপলব্ধ থাকে। এজেন্ট নতুন একটি সম্পূর্ণ আলাপে একই দীর্ঘমেয়াদি জ্ঞান পুনরুদ্ধার করতে পারে।


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## সারাংশ

এই নোটবুকটিতে আপনি একটি কোডিং সহকারী তৈরি করেছিলেন যা **MAF-এর ওয়ার্কিং মেমরি** (`agent.create_session()`) কে **Cognee-এর দীর্ঘস্থায়ী নলেজ গ্রাফের** সাথে সংযুক্ত করে।

### আপনি যা শিখেছেন
1. **জ্ঞান গ্রাফ নির্মাণ**: Cognee অপ্রকৃত পাঠ গ্রহণ করে এবং একটি গ্রাফ + ভেক্টর মেমরি তৈরি করে।
2. **memify দিয়ে গ্রাফ সমৃদ্ধকরণ**: আপনার বিদ্যমান গ্রাফের উপরে প্রাপ্ত তথ্য এবং সমৃদ্ধ সম্পর্ক তৈরি।
3. **MAF + Cognee একাট্টে কাজ করা**: `@tool` ফাংশনগুলো MAF এজেন্টকে Cognee-এর গ্রাফ প্রাকৃতিকভাবে অনুসন্ধান করতে দেয়।
4. **ওয়ার্কিং মেমরি + দীর্ঘস্থায়ী মেমরি**: `AgentSession` (মাধ্যমে `agent.create_session()`) সেশন প্রসঙ্গ প্রদান করে যখন Cognee স্থায়ী জ্ঞান প্রদান করে।
5. **NodeSets দিয়ে ফিল্টার করা অনুসন্ধান**: জ্ঞান গ্রাফের নির্দিষ্ট উপসেট লক্ষ্য করা (যেমন শুধুমাত্র নীতিমালা)।

### মূল বিষয়সমূহ
- **Cognee** কাঁচা পাঠকে কাঠামোবদ্ধ, সম্পর্ক-সচেতন মেমরিতে রূপান্তর করে — যা একটি সাধারণ ভেক্টর স্টোরের চেয়ে বেশি শক্তিশালী।
- **`@tool` ফাংশনগুলো** MAF এজেন্ট এবং বাহ্যিক জ্ঞান সিস্টেমের মধ্যে পরিষ্কার সেতুবন্ধন তৈরি করে।
- **`AgentSession`** (মাধ্যমে `agent.create_session()`) প্রতিটি কথোপকথনের প্রসঙ্গ দীর্ঘস্থায়ী জ্ঞান থেকে আলাদা ভাবে রাখে।
- একই জ্ঞান গ্রাফ একাধিক সেশন এবং এজেন্টদের সেবা দেয়।

### বাস্তব জীবনের ব্যবহার
- **ডেভেলপার কো-পাইলটস**: কোড রিভিউ, ঘটনা বিশ্লেষণ, আর্কিটেকচার সহকারী
- **গ্রাহক-বান্ধব কো-পাইলটস**: পণ্য ডকুমেন্ট, FAQ এবং CRM নোটের ওপর সাপোর্ট এজেন্ট
- **অভ্যন্তরীণ বিশেষজ্ঞ কো-পাইলটস**: নীতি, আইনগত, অথবা নিরাপত্তা সহকারী যারা নির্দেশিকা নিয়ে যুক্তি করবে
- **একীকৃত ডেটা স্তর**: কাঠামোবদ্ধ এবং অপ্রকৃত তথ্য একত্রিত করে একটি অনুসন্ধানযোগ্য গ্রাফ তৈরি

### পরবর্তী ধাপসমূহ
- Cognee-এ কালবৈষম্য সচেতনতার সাথে পরীক্ষা-নিরীক্ষা করা
- ডোমেইন-নির্দিষ্ট গ্রাফ গুণমানের জন্য OWL অন্টোলজি নির্ধারণ করা
- সময়ের সাথে রিট্রিভাল উন্নত করার জন্য ব্যবহারকারী প্রতিক্রিয়া লুপ যুক্ত করা
- একক Cognee মেমরি স্তর ভাগ করে নেওয়া বহু-এজেন্ট সিস্টেমে স্কেল করা


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসাধ্য নির্ভুলতার জন্য চেষ্টা করি, তবে অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসামঞ্জস্য থাকতে পারে। মূল নথিটি তার নিজস্ব ভাষায় সর্বোত্তম authoritative উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদের পরামর্শ দেওয়া হয়। এই অনুবাদের ব্যবহারের ফলে যে কোনো ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
